## 19. Validação multi-região — São Paulo

Após validar o pipeline completo do Sentinela para Minas Gerais, a próxima etapa consiste em aplicar exatamente a mesma metodologia a uma segunda região.

A região selecionada para esta validação é São Paulo (SP).

O objetivo é verificar se:

- o processamento semanal é independente da UF;
- a construção da referência histórica funciona sem regras específicas para MG;
- o modelo de atraso de notificação pode ser estimado por região;
- o nowcast mantém comportamento coerente;
- a máquina de estados operacionais funciona sem alterações específicas;
- os endpoints e o frontend suportam múltiplas regiões reais.

Nenhuma regra estatística será ajustada especificamente para São Paulo nesta etapa.

In [1]:
from pathlib import Path

import pandas as pd


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = project_root / "data" / "raw"

historical_files = sorted(
    raw_data_dir.glob("INFLUD*.csv")
)

print("Arquivos encontrados:", len(historical_files))

for path in historical_files:
    print(path.name)

Arquivos encontrados: 8
INFLUD19-23-03-2026.csv
INFLUD20-23-03-2026.csv
INFLUD21-23-03-2026.csv
INFLUD22-23-03-2026.csv
INFLUD23-23-03-2026.csv
INFLUD24-23-03-2026.csv
INFLUD25-28-09-2026.csv
INFLUD26-28-09-2026.csv


In [2]:
first_week_start_by_year = {
    2019: pd.Timestamp("2018-12-30"),
    2020: pd.Timestamp("2019-12-29"),
    2021: pd.Timestamp("2021-01-03"),
    2022: pd.Timestamp("2022-01-02"),
    2023: pd.Timestamp("2023-01-01"),
    2024: pd.Timestamp("2023-12-31"),
    2025: pd.Timestamp("2024-12-29"),
    2026: pd.Timestamp("2026-01-04"),
}

In [3]:
region = "SP"

regional_counts = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
        ],
        dtype="string",
        low_memory=False,
    )

    normalized_uf = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    count = int(
        normalized_uf.eq(region).sum()
    )

    regional_counts.append(
        {
            "epi_year": year,
            "region": region,
            "record_count": count,
        }
    )

regional_counts_df = pd.DataFrame(
    regional_counts
).sort_values("epi_year")

display(regional_counts_df)

,epi_year,region,record_count
0,2019,SP,12331
1,2020,SP,358612
2,2021,SP,493725
3,2022,SP,160930
4,2023,SP,78490
5,2024,SP,69006
6,2025,SP,79295
7,2026,SP,48978


In [4]:
assert set(
    regional_counts_df["epi_year"]
) == set(range(2019, 2027))

assert (
    regional_counts_df["record_count"]
    > 0
).all()

print(
    "Total SP 2019-2026:",
    regional_counts_df[
        "record_count"
    ].sum(),
)

Total SP 2019-2026: 1301367


In [5]:
def build_regional_weekly(
    historical_files,
    region: str,
    first_week_start_by_year: dict,
) -> pd.DataFrame:
    weekly_frames = []

    region = region.strip().upper()

    for path in historical_files:
        year = int(
            f"20{path.name.upper().split('INFLUD')[1][:2]}"
        )

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
            ],
            dtype="string",
            low_memory=False,
        )

        residence_uf = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        regional_df = df.loc[
            residence_uf.eq(region)
        ].copy()

        regional_df["sympton_date"] = (
            pd.to_datetime(
                regional_df["DT_SIN_PRI"],
                errors="raise",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        first_week_start = (
            first_week_start_by_year[
                year
            ]
        )

        regional_df["epi_week"] = (
            (
                regional_df[
                    "sympton_date"
                ]
                - first_week_start
            ).dt.days
            // 7
            + 1
        )

        assert (
            regional_df["epi_week"]
            .ge(1)
            .all()
        )

        weekly = (
            regional_df
            .groupby("epi_week")
            .size()
            .rename("record_count")
            .reset_index()
        )

        weekly.insert(
            0,
            "epi_year",
            year,
        )

        weekly["region"] = region
        
        weekly_frames.append(
            weekly
        )

    return (
        pd.concat(
            weekly_frames,
            ignore_index=True,
        )
        .sort_values(
            [
                "epi_year",
                "epi_week", 
            ]
        )
        .reset_index(drop=True)
    )

In [6]:
historical_weekly_sp = (
    build_regional_weekly(
        historical_files=historical_files,
        region="SP",
        first_week_start_by_year=(
            first_week_start_by_year
        ),
    )
)

display(
    historical_weekly_sp.groupby(
        "epi_year"
    ).agg(
        weeks=("epi_week", "nunique"),
        total=("record_count", "sum"),
    )
)

,weeks,total
epi_year,,
2019,52,12331
2020,53,358612
2021,52,493725
2022,52,160930
2023,52,78490
2024,52,69006
2025,53,79295
2026,38,48978


In [7]:
def build_reference_before_year(
    historical_weekly: pd.DataFrame,
    target_year: int,
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly["epi_year"] < target_year
    ].copy()

    history = history.loc[
        history["epi_week"].between(1, 52)
    ].copy()

    reference = (
        history
        .groupby("epi_week")["record_count"]
        .agg(
            historical_median="median",
            q25=lambda values: values.quantile(0.25),
            q75=lambda values: values.quantile(0.75),
            historical_years="count",
        )
        .reset_index()
    )

    reference["has_minimum_history"] = (
        reference["historical_years"] >= 3
    )

    return reference

In [8]:
reference_sp_2026 = build_reference_before_year(
    historical_weekly_sp,
    2026,
)

display(reference_sp_2026.head())

,epi_week,historical_median,q25,q75,historical_years,has_minimum_history
0,1,927.0,423.5,5111.0,7,True
1,2,817.0,338.0,5707.5,7,True
2,3,812.0,351.0,5273.0,7,True
3,4,791.0,394.0,4530.0,7,True
4,5,913.0,401.0,3836.0,7,True


In [9]:
weeks_per_year_sp = (
    historical_weekly_sp.loc[
        historical_weekly_sp["epi_year"] < 2026
    ]
    .groupby("epi_year")
    .agg(
        min_week=("epi_week", "min"),
        max_week=("epi_week", "max"),
        weeks=("epi_week", "nunique"),
    )
)

display(weeks_per_year_sp)

,min_week,max_week,weeks
epi_year,,,
2019,1,52,52
2020,1,53,53
2021,1,52,52
2022,1,52,52
2023,1,52,52
2024,1,52,52
2025,1,53,53


In [10]:
assert len(reference_sp_2026) == 52

assert reference_sp_2026["epi_week"].is_unique

assert (
    reference_sp_2026["historical_years"]
    >= 7
).all()

print(
    reference_sp_2026[
        [
            "epi_week",
            "historical_median",
            "q25",
            "q75",
            "historical_years",
        ]
    ].head()
)

print("✓ Referência histórica SP 2019–2025 construída.")

   epi_week  historical_median    q25     q75  historical_years
0         1              927.0  423.5  5111.0                 7
1         2              817.0  338.0  5707.5                 7
2         3              812.0  351.0  5273.0                 7
3         4              791.0  394.0  4530.0                 7
4         5              913.0  401.0  3836.0                 7
✓ Referência histórica SP 2019–2025 construída.


## 5. Cobertura semanal de SP em 2026

A série observada de 2026 será expandida para as 52 semanas epidemiológicas esperadas.

Semanas sem observação não serão preenchidas com zero.

Será feita a distinção entre:

- `observado`: semana com dados disponíveis;
- `em_andamento`: semana atual ainda não consolidada;
- `futura`: semana ainda não alcançada pela data da extração.

Essa separação evita interpretar ausência de cobertura como ausência de casos.

In [11]:
calendar_2026_sp = pd.DataFrame(
    {
        "epi_week": range(1, 53),
    }
)

calendar_2026_sp["epi_year"] = 2026

first_week_start_2026 = (
    first_week_start_by_year[2026]
)

calendar_2026_sp["week_start"] = (
    first_week_start_2026
    + pd.to_timedelta(
        (calendar_2026_sp["epi_week"] - 1) * 7,
        unit="D",
    )
)

calendar_2026_sp["week_end"] = (
    calendar_2026_sp["week_start"]
    + pd.Timedelta(days=6)
)

display(calendar_2026_sp.head())
display(calendar_2026_sp.tail())

,epi_week,epi_year,week_start,week_end
0,1,2026,2026-01-04,2026-01-10
1,2,2026,2026-01-11,2026-01-17
2,3,2026,2026-01-18,2026-01-24
3,4,2026,2026-01-25,2026-01-31
4,5,2026,2026-02-01,2026-02-07


,epi_week,epi_year,week_start,week_end
47,48,2026,2026-11-29,2026-12-05
48,49,2026,2026-12-06,2026-12-12
49,50,2026,2026-12-13,2026-12-19
50,51,2026,2026-12-20,2026-12-26
51,52,2026,2026-12-27,2027-01-02


In [12]:
weekly_sp_2026 = (
    historical_weekly_sp.loc[
        (historical_weekly_sp["epi_year"] == 2026)
        & historical_weekly_sp["epi_week"].between(1, 52)
    ]
    [
        [
            "epi_week",
            "record_count",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

display(weekly_sp_2026.head())
display(weekly_sp_2026.tail())

,epi_week,record_count
0,1,647
1,2,541
2,3,565
3,4,491
4,5,703


,epi_week,record_count
33,34,770
34,35,716
35,36,663
36,37,575
37,38,260


In [13]:
assert len(weekly_sp_2026) == 38

assert weekly_sp_2026[
    "epi_week"
].is_unique

assert (
    weekly_sp_2026[
        "record_count"
    ].sum()
    == 48_978
)

print(
    "Semanas observadas:",
    len(weekly_sp_2026),
)

print(
    "Total SP 2026:",
    weekly_sp_2026[
        "record_count"
    ].sum(),
)

print("✓ Série semanal SP 2026 pronta.")

Semanas observadas: 38
Total SP 2026: 48978
✓ Série semanal SP 2026 pronta.


In [14]:
coverage_sp_2026 = (
    calendar_2026_sp
    .merge(
        weekly_sp_2026,
        on="epi_week",
        how="left",
    )
)

coverage_sp_2026["region"] = "SP"

In [15]:
AS_OF_DATE_2026 = pd.Timestamp(
    "2026-09-28"
)

In [16]:
coverage_sp_2026["calendar_status"] = (
    "futura"
)

coverage_sp_2026.loc[
    coverage_sp_2026["week_end"]
    < AS_OF_DATE_2026,
    "calendar_status",
] = "encerrada"

coverage_sp_2026.loc[
    (
        coverage_sp_2026["week_start"]
        <= AS_OF_DATE_2026
    )
    & (
        coverage_sp_2026["week_end"]
        >= AS_OF_DATE_2026
    ),
    "calendar_status",
] = "em_andamento"

In [17]:
coverage_sp_2026["data_status"] = (
    "sem_cobertura"
)

coverage_sp_2026.loc[
    coverage_sp_2026[
        "record_count"
    ].notna(),
    "data_status",
] = "observado"

In [18]:
coverage_sp_2026[
    "cumulative_record_count"
] = (
    coverage_sp_2026[
        "record_count"
    ]
    .fillna(0)
    .cumsum()
)

coverage_sp_2026.loc[
    coverage_sp_2026[
        "record_count"
    ].isna(),
    "cumulative_record_count",
] = pd.NA


In [19]:
assert len(coverage_sp_2026) == 52

assert coverage_sp_2026[
    "epi_week"
].is_unique

assert (
    coverage_sp_2026.loc[
        coverage_sp_2026[
            "epi_week"
        ].between(1, 38),
        "data_status",
    ]
    .eq("observado")
    .all()
)

assert (
    coverage_sp_2026.loc[
        coverage_sp_2026[
            "epi_week"
        ].between(39, 52),
        "record_count",
    ]
    .isna()
    .all()
)

assert (
    coverage_sp_2026.loc[
        coverage_sp_2026[
            "data_status"
        ].eq("observado"),
        "record_count",
    ]
    .sum()
    == 48_978
)

print("✓ 52 semanas conhecidas.")
print("✓ SE 1–38 observadas.")
print("✓ SE 39–52 preservadas como ausência de cobertura.")
print("✓ Total observado:", coverage_sp_2026["record_count"].sum())

✓ 52 semanas conhecidas.
✓ SE 1–38 observadas.
✓ SE 39–52 preservadas como ausência de cobertura.
✓ Total observado: 48978.0


In [20]:
display(
    coverage_sp_2026[
        [
            "epi_week",
            "week_start",
            "week_end",
            "record_count",
            "calendar_status",
            "data_status",
        ]
    ].tail(15)
)

,epi_week,week_start,week_end,record_count,calendar_status,data_status
37,38,2026-09-20,2026-09-26,260.0,encerrada,observado
38,39,2026-09-27,2026-10-03,NaN,em_andamento,sem_cobertura
39,40,2026-10-04,2026-10-10,NaN,futura,sem_cobertura
40,41,2026-10-11,2026-10-17,NaN,futura,sem_cobertura
41,42,2026-10-18,2026-10-24,NaN,futura,sem_cobertura
42,43,2026-10-25,2026-10-31,NaN,futura,sem_cobertura
43,44,2026-11-01,2026-11-07,NaN,futura,sem_cobertura
44,45,2026-11-08,2026-11-14,NaN,futura,sem_cobertura
45,46,2026-11-15,2026-11-21,NaN,futura,sem_cobertura
46,47,2026-11-22,2026-11-28,NaN,futura,sem_cobertura


In [21]:
signal_evaluation_sp_2026 = (
    coverage_sp_2026
    .merge(
        reference_sp_2026[
            [
               "epi_week",
                "historical_median",
                "q25",
                "q75",
                "historical_years",
                "has_minimum_history", 
            ]
        ],
        on="epi_week",
        how="left",
    )
)

In [22]:
signal_evaluation_sp_2026[
    "median_threshold"
] = (
    signal_evaluation_sp_2026[
        "historical_median"
    ]
    * 1.5
)

signal_evaluation_sp_2026[
    "ratio_to_median"
] = (
    signal_evaluation_sp_2026[
        "record_count"
    ]
    / signal_evaluation_sp_2026[
        "historical_median"
    ]
)

In [23]:
signal_evaluation_sp_2026[
    "above_q75"
] = (
    signal_evaluation_sp_2026[
        "record_count"
    ]
    > signal_evaluation_sp_2026[
        "q75"
    ]
)

signal_evaluation_sp_2026[
    "above_factor_threshold"
] = (
    signal_evaluation_sp_2026[
        "record_count"
    ]
    >= signal_evaluation_sp_2026[
        "median_threshold"
    ]
)

In [24]:
signal_evaluation_sp_2026[
    "signal_status"
] = "NO_SIGNAL"

not_observed = (
    signal_evaluation_sp_2026[
        "data_status"
    ]
    != "observado"
)

insufficient_history = (
    ~signal_evaluation_sp_2026[
        "has_minimum_history"
    ]
)

signal_evaluation_sp_2026.loc[
    not_observed
    | insufficient_history,
    "signal_status",
] = "INSUFFICIENT_DATA"

In [25]:
is_signal = (
    signal_evaluation_sp_2026[
        "data_status"
    ].eq("observado")
    & signal_evaluation_sp_2026[
        "has_minimum_history"
    ].eq(True)
    & signal_evaluation_sp_2026[
        "above_q75"
    ]
    & signal_evaluation_sp_2026[
        "above_factor_threshold"
    ]
)

signal_evaluation_sp_2026.loc[
    is_signal,
    "signal_status",
]= "SIGNAL"

In [26]:
signal_evaluation_sp_2026[
    "signal_reason"
] = (
    "Critérios de aumento incomum "
    "não atendidos."
)

signal_evaluation_sp_2026.loc[
    signal_evaluation_sp_2026[
        "signal_status"
    ].eq("SIGNAL"),
    "signal_reason",
] = (
    "Contagem observada acima de Q75 "
    "e de 1,5× a mediana histórica."
)

signal_evaluation_sp_2026.loc[
    signal_evaluation_sp_2026[
        "signal_status"
    ].eq("INSUFFICIENT_DATA"),
    "signal_reason",
] = (
    "Dados observados ou referência histórica "
    "insuficientes para avaliação." 
)

In [27]:
print(
    signal_evaluation_sp_2026[
        "signal_status"
    ].value_counts()
)

signal_status
NO_SIGNAL            38
INSUFFICIENT_DATA    14
Name: count, dtype: int64


In [28]:
signals_sp_2026 = (
    signal_evaluation_sp_2026.loc[
        signal_evaluation_sp_2026[
            "signal_status"
        ].eq("SIGNAL")
    ]
    [
        [
            "epi_week",
            "record_count",
            "historical_median",
            "q75",
            "median_threshold",
            "ratio_to_median",
            "signal_status",
        ]
    ]
    .reset_index(drop=True)
)

display(signals_sp_2026)

,epi_week,record_count,historical_median,q75,median_threshold,ratio_to_median,signal_status


## 7. Avaliação retrospectiva de sinais — SP 2026

A aplicação das regras estatísticas do Sentinela às 38 semanas observadas de São Paulo em 2026 não identificou sinais de aumento incomum.

### Resultado

- Semanas observadas: 38
- `NO_SIGNAL`: 38
- `SIGNAL`: 0
- `INSUFFICIENT_DATA`: 14

As semanas 39 a 52 foram classificadas como `INSUFFICIENT_DATA` por não possuírem cobertura observada na extração utilizada.

### Interpretação

A ausência de sinais em SP demonstra que a aplicação das mesmas regras utilizadas em Minas Gerais não produz automaticamente alertas em uma região com maior volume absoluto de registros.

Esse resultado é importante para a validação multi-região, pois sugere que os limiares baseados na distribuição histórica semanal acompanham a escala epidemiológica de cada região.

A próxima etapa consiste em incorporar o atraso regional de notificação e avaliar se o nowcast preserva esse comportamento.

In [29]:
model_years_sp = [2023, 2024, 2025]

delay_frames_sp = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if year not in model_years_sp:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    region_uf = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    sp = df.loc[
        region_uf.eq("SP")
    ].copy()

    sp["DT_SIN_PRI"] = pd.to_datetime(
        sp["DT_SIN_PRI"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    sp["DT_NOTIFIC"] = pd.to_datetime(
        sp["DT_NOTIFIC"],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

    sp["delay_notification_days"] = (
        sp["DT_NOTIFIC"]
        - sp["DT_SIN_PRI"]
    ).dt.days

    sp["source_year"] = year

    delay_frames_sp.append(
        sp[
            [
                "source_year",
                "DT_SIN_PRI",
                "DT_NOTIFIC",
                "delay_notification_days",
            ]
        ]
    )

delay_sp = pd.concat(
    delay_frames_sp,
    ignore_index=True,
)

delay_sp = delay_sp.loc[
    delay_sp["delay_notification_days"].notna()
    & delay_sp["delay_notification_days"].ge(0) 
].copy()

print(
    "Registros válidos:",
    len(delay_sp),
)

print(
    "Anos:",
    sorted(
        delay_sp["source_year"].unique()
    ),
)

Registros válidos: 226791
Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]


In [30]:
delay_summary_sp = (
    delay_sp
    .groupby("source_year")[
        "delay_notification_days"
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
        ]
    )
)

display(
    delay_summary_sp[
        [
            "count",
            "mean",
            "25%",
            "50%",
            "75%",
            "90%",
            "95%",
            "max",
        ]
    ]
)

,count,mean,25%,50%,75%,90%,95%,max
source_year,,,,,,,,
2023,78490.0,7.521506,2.0,4.0,7.0,13.0,20.0,750.0
2024,69006.0,8.874025,2.0,4.0,7.0,14.0,25.0,445.0
2025,79295.0,6.997730,2.0,4.0,7.0,12.0,20.0,573.0


In [31]:
completion_rows_sp = []

for year, group in delay_sp.groupby(
    "source_year"
):
    for lag in range(0, 29):
        completeness = (
            group["delay_notification_days"]
            <= lag
        ).mean()

        completion_rows_sp.append(
            {
                "source_year": year,
                "lag_days": lag,
                "completeness": completeness,
            }
        )

completion_by_year_sp = pd.DataFrame(
    completion_rows_sp
)

completion_model_sp = (
    completion_by_year_sp
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_sp[
    "expected_completeness_pct"
] = (
    completion_model_sp[
        "expected_completeness"
    ]
    * 100
)

In [32]:
display(
    completion_model_sp.loc[
        completion_model_sp[
            "lag_days"
        ].isin(
            [0, 7, 14, 21, 28]
        )
    ][
        [
            "lag_days",
            "expected_completeness",
            "expected_completeness_pct",
            "min_completeness",
            "max_completeness",
        ]
    ]
)

,lag_days,expected_completeness,expected_completeness_pct,min_completeness,max_completeness
0,0,0.070690,7.068951,0.070496,0.076073
7,7,0.786024,78.602370,0.759905,0.791525
14,14,0.919098,91.909797,0.905269,0.921912
21,21,0.955752,95.575233,0.941831,0.956441
28,28,0.968094,96.809383,0.954902,0.969308


In [33]:
MIN_COMPLETENESS = 0.80

mature_lag_sp = int(
    completion_model_sp.loc[
        completion_model_sp[
            "expected_completeness"
        ].ge(MIN_COMPLETENESS),
        "lag_days",
    ].min()
)

print(
    "Primeiro lag maduro em SP:",
    f"D+{mature_lag_sp}",
)

Primeiro lag maduro em SP: D+8


## 8. Nowcast regional — SP 2026

A curva histórica de atraso de notificação de São Paulo apresentou maturidade de 80% apenas em D+8.

Por esse motivo, o nowcast regional de SP utilizará:

- modelo de atraso estimado com 2023–2025;
- maturidade mínima de 80%;
- primeiro lag elegível em D+8;
- referência histórica 2019–2025;
- comparação com Q75;
- limiar de 1,5× a mediana histórica.

A análise será limitada às informações disponíveis até a data da extração.

In [34]:
path_2026 = next(
    path
    for path in historical_files
    if "INFLUD26" in path.name.upper()
)

df_sp_2026_temporal = pd.read_csv(
    path_2026,
    sep=";",
    encoding="latin-1",
    usecols=[
        "NU_NOTIFIC",
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_sp_2026_temporal["SG_UF"] = (
    df_sp_2026_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

sp_2026_temporal = df_sp_2026_temporal.loc[
    df_sp_2026_temporal["SG_UF"].eq("SP")
].copy()

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    sp_2026_temporal[column] = pd.to_datetime(
        sp_2026_temporal[column],
        errors="coerce",
        utc=True,
    ).dt.tz_localize(None)

print(
    "Registros SP 2026:",
    len(sp_2026_temporal),
)

Registros SP 2026: 48978


In [35]:
AS_OF_DATE_2026 = pd.Timestamp(
    "2026-09-28"
)

observed_sp_2026 = coverage_sp_2026.loc[
    coverage_sp_2026[
        "data_status"
    ].eq("observado")
].copy()

observed_sp_2026["age_days"] = (
    AS_OF_DATE_2026
    - observed_sp_2026["week_end"]
).dt.days

display(
    observed_sp_2026[
        [
            "epi_week",
            "week_end",
            "age_days",
        ]
    ].tail(10)
)

,epi_week,week_end,age_days
28,29,2026-07-25,65
29,30,2026-08-01,58
30,31,2026-08-08,51
31,32,2026-08-15,44
32,33,2026-08-22,37
33,34,2026-08-29,30
34,35,2026-09-05,23
35,36,2026-09-12,16
36,37,2026-09-19,9
37,38,2026-09-26,2


In [36]:
nowcast_sp_2026_rows = []

for coverage_row in observed_sp_2026.itertuples(
    index=False
):
    epi_week = int(
        coverage_row.epi_week
    )

    week_start = pd.Timestamp(
        coverage_row.week_start
    )

    week_end = pd.Timestamp(
        coverage_row.week_end
    )

    available_age = int(
        (
            AS_OF_DATE_2026
            - week_end
        ).days
    )

    if available_age < 0:
        continue

    reference_row = reference_sp_2026.loc[
        reference_sp_2026[
            "epi_week"
        ].eq(epi_week)
    ]

    if reference_row.empty:
        continue

    reference_row = (
        reference_row.iloc[0]
    )

    historical_median = float(
        reference_row[
            "historical_median"
        ]
    )

    q75 = float(
        reference_row["q75"]
    )

    median_threshold = (
        historical_median * 1.5
    )

    week_cases = sp_2026_temporal.loc[
        sp_2026_temporal[
            "DT_SIN_PRI"
        ].between(
            week_start,
            week_end,
        )
    ].copy()

    max_lag = min(
        available_age,
        28,
    )

    for lag in range(
        0,
        max_lag + 1,
    ):
        completion_row = (
            completion_model_sp.loc[
                completion_model_sp[
                    "lag_days"
                ].eq(lag)
            ]
        )

        if completion_row.empty:
            continue

        expected_completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        if (
            expected_completeness
            < MIN_COMPLETENESS
        ):
            continue

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases[
                    "DT_NOTIFIC"
                ]
                <= snapshot_date
            ).sum()
        )

        nowcast = (
            known_cases
            / expected_completeness
        )

        would_signal = (
            nowcast > q75
            and nowcast
            >= median_threshold
        )

        nowcast_sp_2026_rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date":
                    snapshot_date,
                "known_cases":
                    known_cases,
                "nowcast": nowcast,
                "q75": q75,
                "median_threshold":
                    median_threshold,
                "would_signal":
                    would_signal,
            }
        )

nowcast_sp_2026 = pd.DataFrame(
    nowcast_sp_2026_rows
)

In [37]:
first_signal_sp_2026 = (
    nowcast_sp_2026.loc[
        nowcast_sp_2026[
            "would_signal"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "lag_days",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(first_signal_sp_2026)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,q75,median_threshold,would_signal


In [38]:
if first_signal_sp_2026.empty:
    print(
        "Nenhum SIGNAL maduro identificado "
        "em SP em 2026."
    )
else:
    print(
        "Semanas com SIGNAL maduro:",
        first_signal_sp_2026[
            "epi_week"
        ].tolist(),
    )

    display(
        first_signal_sp_2026[
            [
                "epi_week",
                "lag_days",
                "snapshot_date",
                "known_cases",
                "nowcast",
                "q75",
                "median_threshold",
            ]
        ]
    )

Nenhum SIGNAL maduro identificado em SP em 2026.


In [39]:
observed_sp_2026[
    "is_mature"
] = (
    observed_sp_2026[
        "age_days"
    ]
    >= mature_lag_sp
)

display(
    observed_sp_2026.tail(10)
)

,epi_week,epi_year,week_start,week_end,record_count,region,calendar_status,data_status,cumulative_record_count,age_days,is_mature
28,29,2026,2026-07-19,2026-07-25,1008.0,SP,encerrada,observado,42168.0,65,True
29,30,2026,2026-07-26,2026-08-01,1026.0,SP,encerrada,observado,43194.0,58,True
30,31,2026,2026-08-02,2026-08-08,1044.0,SP,encerrada,observado,44238.0,51,True
31,32,2026,2026-08-09,2026-08-15,913.0,SP,encerrada,observado,45151.0,44,True
32,33,2026,2026-08-16,2026-08-22,843.0,SP,encerrada,observado,45994.0,37,True
33,34,2026,2026-08-23,2026-08-29,770.0,SP,encerrada,observado,46764.0,30,True
34,35,2026,2026-08-30,2026-09-05,716.0,SP,encerrada,observado,47480.0,23,True
35,36,2026,2026-09-06,2026-09-12,663.0,SP,encerrada,observado,48143.0,16,True
36,37,2026,2026-09-13,2026-09-19,575.0,SP,encerrada,observado,48718.0,9,True
37,38,2026,2026-09-20,2026-09-26,260.0,SP,encerrada,observado,48978.0,2,False


## 9. Resultado operacional — SP 2026

A aplicação do nowcast regional de São Paulo não identificou sinais maduros em 2026.

### Resultado

- Semanas observadas: 38
- Semanas maduras: 37
- Semanas imaturas: 1
- `SIGNAL`: 0
- Primeiro lag maduro regional: D+8

A SE 38 possui apenas D+2 desde o encerramento da semana e, portanto, ainda não possui maturidade suficiente para avaliação operacional.

### Interpretação

O pipeline manteve comportamento conservador em São Paulo.

Mesmo com um volume absoluto de registros superior ao observado em Minas Gerais, nenhuma semana madura ultrapassou simultaneamente:

- Q75 da referência histórica;
- 1,5× a mediana histórica.

Isso reforça que o método utiliza o comportamento histórico regional como referência e não apenas o volume absoluto de registros.

In [40]:
signal_week_set_sp_2026 = set(
    first_signal_sp_2026["epi_week"]
)

weekly_status_sp_2026 = (
    observed_sp_2026[
        [
            "epi_week",
            "week_start",
            "week_end",
            "age_days",
            "is_mature",
        ]
    ]
    .copy()
)

weekly_status_sp_2026["has_signal"] = pd.NA

weekly_status_sp_2026.loc[
    weekly_status_sp_2026["is_mature"],
    "has_signal",
] = (
    weekly_status_sp_2026.loc[
        weekly_status_sp_2026["is_mature"],
        "epi_week",
    ]
    .isin(signal_week_set_sp_2026)
)

display(weekly_status_sp_2026.tail(10))

,epi_week,week_start,week_end,age_days,is_mature,has_signal
28,29,2026-07-19,2026-07-25,65,True,False
29,30,2026-07-26,2026-08-01,58,True,False
30,31,2026-08-02,2026-08-08,51,True,False
31,32,2026-08-09,2026-08-15,44,True,False
32,33,2026-08-16,2026-08-22,37,True,False
33,34,2026-08-23,2026-08-29,30,True,False
34,35,2026-08-30,2026-09-05,23,True,False
35,36,2026-09-06,2026-09-12,16,True,False
36,37,2026-09-13,2026-09-19,9,True,False
37,38,2026-09-20,2026-09-26,2,False,<NA>


In [45]:
states_sp_2026 = []

current_state = "NORMAL"

consecutive_signals = 0
consecutive_no_signals = 0

for row in weekly_status_sp_2026.itertuples(
    index=False
):
    epi_week = int(row.epi_week)
    is_mature = bool(row.is_mature)

    if not is_mature:
        states_sp_2026.append(
            {
                "epi_week": epi_week,
                "is_mature": False,
                "has_signal": None,
                "operational_state": "PENDING",
                "last_stable_state": current_state,
                "consecutive_signals": consecutive_signals,
                "consecutive_no_signals": consecutive_no_signals,
            }
        )

        continue

    has_signal = bool(row.has_signal)

    if has_signal:
        consecutive_signals += 1
        consecutive_no_signals = 0

        if current_state == "NORMAL":
            current_state = "SIGNAL"

        elif current_state == "SIGNAL":
            if consecutive_signals >= 2:
                current_state = "ALERT"

    else:
        consecutive_signals = 0
        consecutive_no_signals += 1

        if current_state == "SIGNAL":
            current_state = "NORMAL"

        elif current_state == "ALERT":
            if consecutive_no_signals >= 2:
                current_state = "NORMAL"

    states_sp_2026.append(
        {
            "epi_week": epi_week,
            "is_mature": True,
            "has_signal": has_signal,
            "operational_state": current_state,
            "last_stable_state": current_state,
            "consecutive_signals": consecutive_signals,
            "consecutive_no_signals": consecutive_no_signals,
        }
    )

operational_sp_2026 = pd.DataFrame(
    states_sp_2026
)

display(
    operational_sp_2026.tail(10)
)

,epi_week,is_mature,has_signal,operational_state,last_stable_state,consecutive_signals,consecutive_no_signals
28,29,True,False,NORMAL,NORMAL,0,29
29,30,True,False,NORMAL,NORMAL,0,30
30,31,True,False,NORMAL,NORMAL,0,31
31,32,True,False,NORMAL,NORMAL,0,32
32,33,True,False,NORMAL,NORMAL,0,33
33,34,True,False,NORMAL,NORMAL,0,34
34,35,True,False,NORMAL,NORMAL,0,35
35,36,True,False,NORMAL,NORMAL,0,36
36,37,True,False,NORMAL,NORMAL,0,37
37,38,False,None,PENDING,NORMAL,0,37


In [48]:
assert (
    operational_sp_2026.loc[
        operational_sp_2026["epi_week"].between(1, 37),
        "operational_state",
    ]
    .eq("NORMAL")
    .all()
)

week_38 = operational_sp_2026.loc[
    operational_sp_2026["epi_week"].eq(38)
].iloc[0]

assert week_38["operational_state"] == "PENDING"
assert bool(week_38["is_mature"]) is False
assert week_38["has_signal"] is None

print("✓ SP 2026 permaneceu NORMAL até a SE 37.")
print("✓ SE 38 corretamente classificada como PENDING.")

✓ SP 2026 permaneceu NORMAL até a SE 37.
✓ SE 38 corretamente classificada como PENDING.


In [49]:
nowcast_sp_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_sp_nowcast.csv"
)

operational_sp_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_sp_operational_status.csv"
)

nowcast_sp_2026.to_csv(
    nowcast_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

operational_sp_2026.to_csv(
    operational_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

print("✓", nowcast_sp_2026_path)
print("✓", operational_sp_2026_path)

✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_sp_nowcast.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_sp_operational_status.csv


## 10. Validação externa — SP 2026

A ausência de sinais maduros identificados pelo Sentinela em São Paulo foi comparada com evidências epidemiológicas oficiais disponíveis para 2026.

Até a SE 16, a vigilância estadual havia registrado 10.467 hospitalizações por SRAG, das quais 1.238 apresentaram detecção de influenza. Apesar da carga absoluta relevante, as fontes oficiais consultadas não indicaram, no período analisado, uma ruptura estadual comparável ao episódio de Minas Gerais em 2025.

Em abril de 2026, o Governo do Estado de São Paulo destacou maior circulação sazonal de vírus respiratórios e reforçou a vacinação contra influenza.

Entretanto, boletins de vigilância do período não colocavam São Paulo entre as unidades federativas do Sudeste com incidência de SRAG em nível de alerta e tendência de crescimento, situação observada naquele momento em Minas Gerais e Espírito Santo.

### Interpretação

O resultado fornece um teste negativo importante para o Sentinela.

O sistema:

- identificou um episódio retrospectivamente relevante em MG;
- não produziu sinais maduros em SP/2026;
- não reagiu apenas ao elevado volume absoluto de casos;
- manteve a comparação baseada no comportamento histórico específico da região.

Esses achados são compatíveis com a hipótese de que os limiares regionais ajudam a distinguir carga sazonal esperada de aumentos epidemiológicos incomuns.

In [50]:
processed_dir = (
    project_root
    / "data"
    / "processed"
)

historical_weekly_sp_path = (
    processed_dir
    / "srag_sp_historical_weekly_2019_2026.csv"
)

reference_sp_2026_path = (
    processed_dir
    / "srag_sp_historical_reference_2019_2025.csv"
)

coverage_sp_2026_path = (
    processed_dir
    / "srag_2026_sp_weekly_coverage.csv"
)

signals_sp_2026_path = (
    processed_dir
    / "srag_2026_sp_signal_evaluation.csv"
)

historical_weekly_sp.to_csv(
    historical_weekly_sp_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

reference_sp_2026.to_csv(
    reference_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

coverage_sp_2026.to_csv(
    coverage_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

signal_evaluation_sp_2026.to_csv(
    signals_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

print("✓", historical_weekly_sp_path)
print("✓", reference_sp_2026_path)
print("✓", coverage_sp_2026_path)
print("✓", signals_sp_2026_path)

✓ /home/matheuspessoa/Sentinela/data/processed/srag_sp_historical_weekly_2019_2026.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_sp_historical_reference_2019_2025.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_sp_weekly_coverage.csv
✓ /home/matheuspessoa/Sentinela/data/processed/srag_2026_sp_signal_evaluation.csv


In [51]:
weekly_sp_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_sp_weekly.csv"
)

weekly_sp_2026.to_csv(
    weekly_sp_2026_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

print(
    "✓ Série semanal SP 2026 salva em:",
    weekly_sp_2026_path,
)

✓ Série semanal SP 2026 salva em: /home/matheuspessoa/Sentinela/data/processed/srag_2026_sp_weekly.csv
